In [1]:
import os
import zipfile
import requests
import tempfile
import joblib
import pandas as pd
from models._utils import amae
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer
from sklearn.model_selection import StratifiedKFold

In [2]:
url = "https://www.uco.es/grupos/ayrna/datasets/TOC-UCO.zip"
response = requests.get(url)

temp_file = tempfile.NamedTemporaryFile(delete=False)
temp_file.write(response.content)

tmp_tocuco_path = tempfile.mkdtemp()
zip_ref = zipfile.ZipFile(temp_file.name, 'r')
zip_ref.extractall(tmp_tocuco_path)
extracted_files = zip_ref.namelist()

tmp_tocuco_path = os.path.join(tmp_tocuco_path, "TOC-UCO")

print(f"TOC-UCO repository saved temporarily at {tmp_tocuco_path}")
print(f"Files inside the repository: {os.listdir(tmp_tocuco_path)}")

TOC-UCO repository saved temporarily at /tmp/tmp_3tbmcd4/TOC-UCO
Files inside the repository: ['train_masks.pkl', 'metadata.csv', 'data', 'train_masks.json']


In [ ]:
from models.xgbord import XGBORD

SEEDS = 2 #! Seeds should be <= 30, in the paper we consider 30 seeds

## Get training masks to make the train/test partitions
with open(os.path.join(tmp_tocuco_path, "train_masks.pkl"), "rb") as train_masks_binary:
    train_masks = joblib.load(train_masks_binary)

tocuco_datasets_path = os.path.join(tmp_tocuco_path, "data")

xgbord_params = {
    "max_depth": [3, 5, 8],
    "n_estimators": [100, 250, 500, 1000],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.75, 0.95, 1.0],
    "colsample_bytree": [0.75, 0.95, 1.0],
}

# initialize cv parameters
amae_scorer = make_scorer(amae, greater_is_better=False)
random_search_n_iters = 2 #! In the paper we consider 20 iterations
random_search_cv_method = StratifiedKFold(n_splits=3)

for dataset_name in os.listdir(tocuco_datasets_path):
    dataset = pd.read_csv(os.path.join(tocuco_datasets_path, dataset_name))
    for seed in range(SEEDS):
        dataset_name_without_extension = dataset_name.split(".")[0]

        # Get train mask for the current dataset and seed
        dataset_seed_train_mask = train_masks[f"{dataset_name_without_extension}_seed_{seed}"]

        # Apply the mask to get train and test partitions
        train = dataset.loc[dataset_seed_train_mask]
        test = dataset.loc[~dataset_seed_train_mask]

        # Separate features and target variable and convert to numpy arrays
        X_train = train.drop(columns=["y"]).to_numpy()
        X_test = test.drop(columns=["y"]).to_numpy()
        y_train = train["y"].to_numpy()
        y_test = test["y"].to_numpy()
        del train, test

        models = {
            "xgbord_baseline_ucw": RandomizedSearchCV(
                XGBORD(thresholds_method="mean", balanced_class_weights=False, random_state=seed),
                scoring=amae_scorer,
                param_distributions=xgbord_params,
                n_iter=random_search_n_iters,
                cv=random_search_cv_method,
                n_jobs=1,
                verbose=3,
                random_state=seed,
                error_score=("raise"),
            ),
            "xgbord_baseline_bcw": RandomizedSearchCV(
                XGBORD(thresholds_method="mean", balanced_class_weights=True, random_state=seed),
                scoring=amae_scorer,
                param_distributions=xgbord_params,
                n_iter=random_search_n_iters,
                cv=random_search_cv_method,
                n_jobs=1,
                verbose=3,
                random_state=seed,
                error_score=("raise"),
            ),
            "xgbord_lat_med_ucw": RandomizedSearchCV(
                XGBORD(thresholds_method="projections", balanced_class_weights=False, random_state=seed),
                scoring=amae_scorer,
                param_distributions=xgbord_params,
                n_iter=random_search_n_iters,
                cv=random_search_cv_method,
                n_jobs=1,
                verbose=3,
                random_state=seed,
                error_score=("raise"),
            ),
            "xgbord_lat_med_bcw": RandomizedSearchCV(
                XGBORD(thresholds_method="projections", balanced_class_weights=True, random_state=seed),
                scoring=amae_scorer,
                param_distributions=xgbord_params,
                n_iter=random_search_n_iters,
                cv=random_search_cv_method,
                n_jobs=1,
                verbose=3,
                random_state=seed,
                error_score=("raise"),
            ),
            "xgbord_imp_med_ucw": RandomizedSearchCV(
                XGBORD(thresholds_method="median", balanced_class_weights=False, random_state=seed),
                scoring=amae_scorer,
                param_distributions=xgbord_params,
                n_iter=random_search_n_iters,
                cv=random_search_cv_method,
                n_jobs=1,
                verbose=3,
                random_state=seed,
                error_score=("raise"),
            ),
            "xgbord_imp_med_bcw": RandomizedSearchCV(
                XGBORD(thresholds_method="median", balanced_class_weights=True, random_state=seed),
                scoring=amae_scorer,
                param_distributions=xgbord_params,
                n_iter=random_search_n_iters,
                cv=random_search_cv_method,
                n_jobs=1,
                verbose=3,
                random_state=seed,
                error_score=("raise"),
            ),
            "xgbord_opt_thresh_ucw": RandomizedSearchCV(
                XGBORD(thresholds_method="optimal", balanced_class_weights=False, random_state=seed),
                scoring=amae_scorer,
                param_distributions=xgbord_params,
                n_iter=random_search_n_iters,
                cv=random_search_cv_method,
                n_jobs=1,
                verbose=3,
                random_state=seed,
                error_score=("raise"),
            ),
            "xgbord_opt_thresh_bcw": RandomizedSearchCV(
                XGBORD(thresholds_method="optimal", balanced_class_weights=True, random_state=seed),
                scoring=amae_scorer,
                param_distributions=xgbord_params,
                n_iter=random_search_n_iters,
                cv=random_search_cv_method,
                n_jobs=1,
                verbose=3,
                random_state=seed,
                error_score=("raise"),
            )
        }

        for model_name, model in models.items():
            model.fit(X_train, y_train)
            best_adabord = model.best_estimator_
            y_pred = best_adabord.predict(X_test)
            balanced_acc = balanced_accuracy_score(y_test, y_pred)
            print(f"Dataset: {dataset_name_without_extension}, Seed: {seed}, Model {model_name}, Balanced Accuracy: {balanced_acc}, AMAE: {amae(y_test, y_pred)}")

Fitting 3 folds for each of 2 candidates, totalling 6 fits
Using uniform class weights.
Computing thresholds using mean projections...
Threshold method: mean thresholds: [ 0.14232042 -0.0519716   1.0828301 ]
[CV 1/3] END colsample_bytree=0.95, learning_rate=0.05, max_depth=8, n_estimators=250, subsample=0.95;, score=-1.097 total time=   9.5s
Using uniform class weights.
Computing thresholds using mean projections...
Threshold method: mean thresholds: [-0.7236687  -0.01544785  0.4283832 ]
[CV 2/3] END colsample_bytree=0.95, learning_rate=0.05, max_depth=8, n_estimators=250, subsample=0.95;, score=-1.049 total time=  14.4s
Using uniform class weights.
Computing thresholds using mean projections...
Threshold method: mean thresholds: [0.35667244 0.3424314  1.0073345 ]
[CV 3/3] END colsample_bytree=0.95, learning_rate=0.05, max_depth=8, n_estimators=250, subsample=0.95;, score=-1.090 total time=  14.7s
Using uniform class weights.
